In [ ]:
Imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
import pdfplumber
import os
from dotenv import load_dotenv

In [ ]:
# Add retry logic and caching
import time
import hashlib
from tenacity import retry, stop_after_attempt, wait_exponential

# Simple cache for responses
response_cache = {}

def get_cached_response(question):
    """Check if we have cached response"""
    key = hashlib.md5(question.encode()).hexdigest()
    return response_cache.get(key)

def cache_response(question, response):
    """Cache the response"""
    key = hashlib.md5(question.encode()).hexdigest()
    response_cache[key] = response

In [36]:
# Import YouTube recommender
from youtube_recommender import YouTubeRecommender

In [ ]:
# Extract text from PDFs
pdf_input = r'D:\CourseAssist\config\data\Class_11_Physics_Ch4.pdf'
all_text = ""

if os.path.isdir(pdf_input):
    pdf_files = [f for f in os.listdir(pdf_input) if f.lower().endswith('.pdf')]
    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in directory {pdf_input!r}")
    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_input, pdf_file)
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                all_text += page.extract_text() or ""
                all_text += "\n"
elif os.path.isfile(pdf_input):
    with pdfplumber.open(pdf_input) as pdf:
        for page in pdf.pages:
            all_text += page.extract_text() or ""
            all_text += "\n"
else:
    raise FileNotFoundError(f"Provided path is neither a directory nor a file: {pdf_input!r}")

In [ ]:
# Create chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([all_text])
print(f"Created {len(chunks)} chunks")

Created 183 chunks


In [46]:
chunks[10]

Document(metadata={}, page_content='would hold from common experience. Even a\nA ball released from rest on one of the planes rolls\nsmall child playing with a simple (non-electric)\ndown and climbs up the other. If the planes are\ntoy-car on a floor knows intuitively that it needs\nsmooth, the final height of the ball is nearly the\nto constantly drag the string attached to the toy-\nsame as the initial height (a little less but never\ncar with some force to keep it going. If it releases')

In [ ]:
# Create vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [53]:
# Show first 10 embeddings from vector store
index_to_docstore = vector_store.index_to_docstore_id

first_10_embeddings = dict(list(index_to_docstore.items())[:10])
print(f"First 10 embeddings (out of {len(index_to_docstore)} total):")
for idx, doc_id in first_10_embeddings.items():
    print(f"  Index {idx}: {doc_id}")

First 10 embeddings (out of 183 total):
  Index 0: 70f65b0f-7a9b-4f19-b7cf-27ebb0858d3f
  Index 1: 75a4387a-0fd2-4ada-bfd8-dd3a387e1a69
  Index 2: 3c9d1d16-8a05-42fe-bd7f-b30405b9adae
  Index 3: 3adaafe9-6df7-4900-87f8-89df7045e671
  Index 4: 0afaab01-06ab-4b8d-bf10-97e94569f41e
  Index 5: 36ccd5a7-9c4a-484f-981d-26270f68327b
  Index 6: 94f97ab7-bb35-4c66-a955-1cae8b934a5e
  Index 7: 66b403c2-7d95-4602-86d1-c56d5d8f54ec
  Index 8: 7acdb29c-5a02-4085-8bd2-762073b23529
  Index 9: 74ce07f3-768c-410b-b9e4-9be1bd8a42b4


In [ ]:
# Initialize LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.2
)

In [41]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=['context', 'question']
)

In [ ]:
# Initialize YouTube Recommender
youtube_recommender = YouTubeRecommender(
    api_key=YOUTUBE_API_KEY,
    embedding_model_name="all-MiniLM-L6-v2"
)

In [ ]:
# Complete RAG Pipeline with YouTube Integration (with retry & caching)

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=4, max=15))
def answer_question_with_videos(question: str):
    """
    Complete pipeline that answers question and recommends videos.
    Includes retry logic and response caching.
    """
    # Check cache first
    cached = get_cached_response(question)
    if cached:
        print("✅ Using cached response...")
        return cached
    
    # Add throttle to avoid rate limits
    time.sleep(2)
    
    